# Backtesting Engine — MACD Crossover Demo

Dieses Notebook demonstriert einen vollständigen Backtest mit der
`packages.backtesting` Engine. Es generiert automatisch synthetische
OHLCV-Kerzen, wenn keine echte CSV-Datei vorhanden ist.

**Abhängigkeiten**: `pyyaml`, `pandas`, `matplotlib` (optional für Charts)

In [ ]:
# Cell 1: Imports & Setup
from __future__ import annotations

import sys
from datetime import datetime, timedelta

import numpy as np
import yaml

from packages.backtesting import (
    BacktestConfig,
    BacktestEngine,
    BacktestReport,
    Candle,
    CsvDataFeed,
    MACDCrossover,
)
from packages.backtesting.datafeed import MemoryDataFeed

print("✓ Alle Imports erfolgreich")

In [ ]:
# Cell 2: Config laden (BacktestConfig)
config_dict = {
    "initial_capital": 100000,
    "max_position_size": 0.25,
    "warmup_bars": 50,
    "commission_rate": 0.001,
    "slippage_bps": 5.0,
    "allow_short": True,
    "symbol": "BTC/USDT",
    "timeframe": "1d",
}

config = BacktestConfig(**config_dict)
print(f"Config: {config.symbol} | Capital: ${config.initial_capital:,.2f} | Timeframe: {config.timeframe}")

In [ ]:
# Cell 3: DataFeed erstellen (mit synthetischem Fallback)
data_path = "data/btc_usdt_1d.csv"

try:
    data_feed = CsvDataFeed(filepath=data_path, symbol=config.symbol)
    candles = data_feed.get_candles()
    print(f"✓ Echt-Daten geladen: {len(candles)} Kerzen aus {data_path}")
except FileNotFoundError:
    print("⚠ CSV nicht gefunden → generiere synthetische Kerzen...")

    def generate_synthetic_candles(n: int = 365, symbol: str = "BTC/USDT", start_price: float = 30000) -> list:
        """Generate synthetic OHLCV candles for demo."""
        np.random.seed(42)
        base = datetime(2024, 1, 1)
        candles = []
        price = start_price
        for i in range(n):
            dt = base + timedelta(days=i)
            ret = np.random.normal(0.0001, 0.02)
            o = price
            c = price * (1 + ret)
            h = max(o, c) * (1 + abs(np.random.normal(0, 0.01)))
            l = min(o, c) * (1 - abs(np.random.normal(0, 0.01)))
            v = np.random.uniform(100, 1000)
            candles.append(Candle(
                timestamp=dt, symbol=symbol, open=o, high=h, low=l, close=c, volume=v
            ))
            price = c
        return candles

    candles = generate_synthetic_candles(n=365, symbol=config.symbol, start_price=30000)
    data_feed = MemoryDataFeed(candles=candles)
    print(f"✓ Synthetische Kerzen: {len(candles)} Bares")

In [ ]:
# Cell 4: MACDCrossover Strategy instantiieren
strategy = MACDCrossover(
    name="macd_12_26_9",
    fast_period=12,
    slow_period=26,
    signal_period=9,
    min_confidence=0.6,
    position_size=0.1,
)
print(f"✓ Strategie: {strategy.name} | Position Size: {strategy.position_size}")

In [ ]:
# Cell 5: BacktestEngine laufen lassen
engine = BacktestEngine(config=config)
result = engine.run(data_feed, strategy)

print(f"✓ Backtest abgeschlossen")
print(f"  Kerzen verarbeitet: {result.metadata.get('candles_processed', 0)}")
print(f"  Trades: {result.total_trades}")
print(f"  Equity Curve Punkte: {len(result.metadata.get('equity_curve', []))}")

In [ ]:
# Cell 6: Key Metrics tabellarisch anzeigen
metrics = result.metrics

metrics_display = {
    "Metric": [
        "Initial Capital",
        "Final Equity",
        "Total Return","Sharpe Ratio",
        "Sortino Ratio",
        "Max Drawdown",
        "Win Rate",
        "Profit Factor",
        "Total Trades",
    ],
    "Value": [
        f"${config.initial_capital:,.2f}",
        f"${result.final_equity:,.2f}",
        f"{metrics.get('total_return', 0)*100:.2f}%",
        f"{metrics.get('sharpe_ratio', 0):.3f}",
        f"{metrics.get('sortino_ratio', 0):.3f}",
        f"{metrics.get('max_drawdown', 0)*100:.2f}%",
        f"{metrics.get('win_rate', 0)*100:.1f}%",
        f"{metrics.get('profit_factor', 0):.3f}",
        str(result.total_trades),
    ],
}

try:
    import pandas as pd
    df = pd.DataFrame(metrics_display)
    display(df)
except ImportError:
    print("pandas nicht verfügbar → Tabellarische Ausgabe:")
    for k, v in zip(metrics_display["Metric"], metrics_display["Value"]):
        print(f"  {k:25s} {v:>15s}")

In [ ]:
# Cell 7: Equity Curve als matplotlib Chart
import matplotlib.pyplot as plt

equity_curve = result.metadata.get("equity_curve", [])
snapshots = result.snapshots

if equity_curve and snapshots:
    dates = [s.timestamp for s in snapshots]

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(dates, equity_curve, linewidth=1.2)
    ax.set_title("Equity Curve — MACD Crossover Backtest", fontsize=14)
    ax.set_xlabel("Datum")
    ax.set_ylabel("Equity ($)")
    ax.grid(True, alpha=0.3)
    ax.ticklabel_format(style="plain", axis="y")

    # Markieren Anfang/Ende
    ax.annotate(
        f"${equity_curve[0]:,.0f}",
        xy=(dates[0], equity_curve[0]),
        fontsize=9,
        color="blue",
    )
    ax.annotate(
        f"${equity_curve[-1]:,.0f}",
        xy=(dates[-1], equity_curve[-1]),
        fontsize=9,
        color="green",
    )
    plt.tight_layout()
    plt.show()
else:
    print("Keine Equity Curve Daten verfügbar")

In [ ]:
# Cell 8: Trade Log als DataFrame
try:
    import pandas as pd

    if result.trades:
        trade_rows = []
        for t in result.trades:
            trade_rows.append({
                "trade_id": t.trade_id,
                "instrument": t.instrument,
                "side": t.side,
                "quantity": round(t.quantity, 6),
                "price": round(t.price, 2),
                "commission": round(t.commission, 2),
                "pnl": round(t.pnl, 2),
                "timestamp": t.timestamp.isoformat(),
            })
        df_trades = pd.DataFrame(trade_rows)
        display(df_trades)
        print(f"\n{len(df_trades)} Trades gelistet")
    else:
        print("Keine Trades im Ergebnis (vielleicht kein Crossover während der Handelsphase)")

    # Trade-Zusammenfassung
    print(f"\n{'='*50}")
    print("TRADE ZUSAMMENFASSUNG")
    print(f"{'='*50}")
    print(f"  Gesamt Trades:          {result.total_trades}")
    print(f"  Verarbeitete Kerzen:    {result.metadata.get('candles_processed', 0)}")
    print(f"  Anfangs-Kapital:        ${config.initial_capital:,.2f}")
    print(f"  End-Equity:             ${result.final_equity:,.2f}")
    pnl_total = sum(t.pnl for t in result.trades)
    print(f"  Kumuliierter P&L:       ${pnl_total:,.2f}")
    print(f"{'='*50}")

except ImportError:
    print("pandas nicht verfügbar")
    print(f"Trades: {result.total_trades}")
    for t in result.trades[:5]:
        print(f"  {t.trade_id} {t.side:>4s} {t.quantity:.6f} @ ${t.price:.2f} P&L=${t.pnl:.2f}")